# 01.01 OpenAI / Azure OpenAI

OpenAI의 관리형 API는 LangChain에서 가장 먼저 만나는 공급자다. `ChatOpenAI`·`AzureChatOpenAI`·`init_chat_model("openai:…")` 3가지 진입점과 GPT-4.1 이후 도입된 **Responses API**(`use_responses_api=True`) 옵션을 다룬다.

## 학습 목표

- `ChatOpenAI` 직접 인스턴스화 vs `init_chat_model("openai:<model>")` v1 헬퍼 경로를 구분한다
- `AzureChatOpenAI`에서 `azure_endpoint`·`api_version`·`azure_deployment` 세 축 매핑을 이해한다
- Responses API(`use_responses_api=True, store=False, include=["reasoning.encrypted_content"]`)로 데이터 보존 없이 reasoning summary를 받는다
- `model.profile` 속성으로 `max_input_tokens`·`tool_calling`·`reasoning_output` 능력치를 조회한다

## 언제 쓰나

- GPT-4.1 / GPT-4o / o-series 등 OpenAI 최신 모델을 사용하는 모든 에이전트
- Microsoft Entra / Azure 구독에 묶인 엔터프라이즈 배포(Azure OpenAI 리소스)
- **Zero Data Retention** 요구가 있는 환경에서 reasoning trace를 암호화 포맷으로 수신해야 할 때

## 01.01.1 환경 설정

`langchain-openai` 패키지와 `.env`의 `OPENAI_API_KEY`(Azure의 경우 `AZURE_OPENAI_*`) 가 필요하다.

In [ ]:
# !pip install -U langchain langchain-openai
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

## 01.01.2 기본 사용 — `ChatOpenAI`

`ChatOpenAI(model="gpt-4.1-mini")` 한 줄로 chat model을 만든다. `invoke()`는 단일 AIMessage를, `stream()`은 토큰 단위 제너레이터를 반환한다.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
msg = model.invoke("LangChain v1에서 init_chat_model의 역할을 한 문장으로 설명해줘.")
print(msg.content)
print("usage:", msg.usage_metadata)

## 01.01.3 `init_chat_model()` — v1 권장 진입점

`init_chat_model("<provider>:<model>")` 는 프로바이더 문자열 하나로 어떤 chat 모델이든 교체할 수 있게 한다. 구성 가능(`configurable`) 파라미터로 프로덕션에서 모델 A/B 테스트도 쉽다.

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model("openai:gpt-4.1-mini", temperature=0)
print(type(m).__name__, "—", m.invoke("안녕").content)

# configurable_fields로 런타임 모델 스왑
configurable = init_chat_model("openai:gpt-4.1-mini", configurable_fields=("model",))
out = configurable.invoke("한 단어로 인사해줘.", config={"configurable": {"model": "gpt-4.1-nano"}})
print("swap →", out.content)

## 01.01.4 `.profile` 속성으로 능력치 조회

`langchain-core` 1.0+ chat 모델은 `.profile` 속성을 통해 컨텍스트 길이·tool calling·구조화 출력·reasoning 지원 여부를 런타임에 알려준다. 에이전트 팩토리 코드에서 **"이 모델이 tool calling을 지원하는가"**를 결정할 때 쓴다.

In [ ]:
profile = model.profile
print("max_input_tokens :", profile.get("max_input_tokens"))
print("max_output_tokens:", profile.get("max_output_tokens"))
print("tool_calling    :", profile.get("tool_calling"))
print("structured_output:", profile.get("structured_output"))
print("reasoning_output :", profile.get("reasoning_output"))

## 01.01.5 Responses API — reasoning + zero retention

GPT-5 / o-series reasoning 모델을 OpenAI Responses 엔드포인트로 호출하려면 `use_responses_api=True`를 준다. `store=False`는 **OpenAI 서버에 응답을 저장하지 않음**(ZDR), `include=["reasoning.encrypted_content"]`는 암호화된 reasoning을 응답에 동봉해 다음 호출에 다시 넣을 수 있게 한다.

In [ ]:
reasoning = ChatOpenAI(
    model="gpt-5-mini",
    use_responses_api=True,
    store=False,
    include=["reasoning.encrypted_content"],
)
r = reasoning.invoke("3단계로 나눠서 17의 제곱근을 추정해줘.")
print("content[:200]:", r.content[:200] if isinstance(r.content, str) else r.content)
# response_metadata 에 reasoning summary가 포함된다
print("reasoning keys:", list((r.response_metadata or {}).keys()))

## 01.01.6 `AzureChatOpenAI` — 엔터프라이즈 배포

Azure OpenAI는 **리소스 엔드포인트 + API 버전 + 배포(deployment) 이름** 3축이 필요하다. 환경변수로 넣어 두면 생성자는 생략 가능하다. (이 셀은 자격이 없으면 그냥 건너뛰어도 된다 — 아래 조건 블록은 key 유무만 확인한다.)

In [ ]:
from langchain_openai import AzureChatOpenAI

have_azure = all(os.environ.get(k) for k in ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_API_VERSION"])
print("Azure 자격 감지:", have_azure)

if have_azure:
    azure = AzureChatOpenAI(
        azure_deployment=os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1-mini"),
        api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    )
    print("azure →", azure.invoke("ping").content[:50])
else:
    print("자격 없음 — Azure 경로는 스킵. ChatOpenAI 경로만 사용해도 무방.")

## 01.01.7 Tool calling + structured output

`bind_tools([...])`로 함수 호출을 활성화하고, `with_structured_output(Schema)`로 Pydantic 모델 구조를 강제한다. OpenAI는 v1 에이전트에서 가장 호환성이 좋은 기본값이다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

@tool
def get_price(ticker: str) -> float:
    """티커의 가격을 반환한다."""
    return {"AAPL": 210.3, "GOOG": 180.1}.get(ticker.upper(), 0.0)

tool_model = model.bind_tools([get_price])
resp = tool_model.invoke("AAPL 가격 얼마야?")
print("tool_calls:", resp.tool_calls)

class Quote(BaseModel):
    ticker: str = Field(..., description="심볼")
    price: float

structured = model.with_structured_output(Quote)
print("structured →", structured.invoke("GOOG가 180.1달러야. 구조화해줘."))

## 체크리스트

- [ ] `ChatOpenAI` 직접 생성 / `init_chat_model("openai:…")` / `AzureChatOpenAI` 3경로를 구분했다
- [ ] `.profile`에서 `tool_calling`·`reasoning_output`·`max_input_tokens`를 확인했다
- [ ] Responses API(`use_responses_api=True`) + `store=False`의 ZDR 용도를 이해했다
- [ ] `include=["reasoning.encrypted_content"]`가 필요한 reasoning trace 재주입 시나리오를 식별했다

## 다음

- `02_anthropic.ipynb` — Claude extended thinking · tool calling
- `11_provider_middleware/07_openai_moderation.ipynb` — OpenAI moderation middleware

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/openai
- OpenAI Responses API: https://platform.openai.com/docs/guides/responses